# FERRUS OSSEUS — Fregate 04
## FLOTTE FERRUS | AD MAJOREM GLORIAM IMPERATORIS

**Pipeline** : Mesh T-pose (sans rig) → FBX rigé (R15 | Mixamo | DeepMotion)

```
Drive/FLOTTE-FERRUS/04_FERRUS_OSSEUS/IN/   <- avatar brut .glb/.gltf/.obj/.fbx
Drive/FLOTTE-FERRUS/04_FERRUS_OSSEUS/OUT/  <- avatar_rigged_*.fbx + rapport
```

**Prerequis** : avatar en T-pose (bras horizontaux, jambes droites). Sans squelette.

In [ ]:
# ══════════════════════════════════════════════════════════════════════════
# CELLULE 1 — CONFIGURATION
# Editer uniquement cette cellule avant de lancer
# ══════════════════════════════════════════════════════════════════════════

# ── Fichier source ────────────────────────────────────────────────────────
INPUT_FILE = "avatar_P1.glb"    # Nom du fichier dans IN/
                                 # Formats supportes : .glb .gltf .obj .fbx

# ── Template squelette ───────────────────────────────────────────────────
TEMPLATE = "r15"                 # r15 | mixamo | deepmotion
#   r15        : 15 bones Roblox R15       → pour FERRUS ANIMUS (Roblox)
#   mixamo     : 26 bones Mixamo           → pour Adobe Mixamo / Unity
#   deepmotion : 52 bones DeepMotion       → pour FERRUS ANIMUS (DeepMotion)

# ── Google Drive ─────────────────────────────────────────────────────────
# Toutes les donnees (IN, OUT, codebase) sont sur Drive pour persistance.
# Les fichiers survivent entre les sessions Colab.
DRIVE_BASE = "/content/drive/MyDrive/FLOTTE-FERRUS"

# ── Repo GitHub ──────────────────────────────────────────────────────────
GITHUB_REPO  = "kioka8877-ux/FLOTTE-FERRUS"
GITHUB_TOKEN = ""   # Token pour git pull/push (repo prive)
                    # Laisser vide si repo public

# ── Paths derives (ne pas modifier) ─────────────────────────────────────
FREGATE_DIR = f"{DRIVE_BASE}/04_FERRUS_OSSEUS"
IN_DIR      = f"{FREGATE_DIR}/IN"
OUT_DIR     = f"{FREGATE_DIR}/OUT"
CODE_DIR    = f"{FREGATE_DIR}/codebase"
SCRIPT_PATH = f"{CODE_DIR}/osseus_core.py"

print("[OSSEUS] Configuration :")
print(f"  Input    : {INPUT_FILE}")
print(f"  Template : {TEMPLATE}")
print(f"  Drive    : {DRIVE_BASE}")
print(f"  IN       : {IN_DIR}")
print(f"  OUT      : {OUT_DIR}")

In [ ]:
# ══════════════════════════════════════════════════════════════════════════
# CELLULE 2 — MONTAGE GOOGLE DRIVE
# Persistance des IN/, OUT/ et du codebase entre sessions
# ══════════════════════════════════════════════════════════════════════════

from google.colab import drive
import os

print("[OSSEUS] Montage Google Drive...")
drive.mount("/content/drive")

# Creer la structure sur Drive si premiere utilisation
for d in [FREGATE_DIR, IN_DIR, OUT_DIR, CODE_DIR, f"{CODE_DIR}/docs"]:
    os.makedirs(d, exist_ok=True)

print(f"[OSSEUS] Drive monte. Structure :")
print(f"  {FREGATE_DIR}/")
print(f"    IN/    <- deposer l'avatar ici")
print(f"    OUT/   <- les FBX rigues apparaissent ici")
print(f"    codebase/")

# Lister le contenu de IN/
in_files = os.listdir(IN_DIR) if os.path.exists(IN_DIR) else []
if in_files:
    print(f"\n[OSSEUS] IN/ contient : {in_files}")
else:
    print(f"\n[OSSEUS] IN/ est vide — deposer un avatar avant de continuer")

In [ ]:
# ══════════════════════════════════════════════════════════════════════════
# CELLULE 3 — SYNCHRONISATION CODEBASE DEPUIS GITHUB
# Recupere toujours la derniere version du code en production
# ══════════════════════════════════════════════════════════════════════════

import subprocess, os

def run(cmd, cwd=None, silent=False):
    result = subprocess.run(cmd, shell=True, capture_output=True, text=True, cwd=cwd)
    if not silent:
        if result.stdout.strip(): print(result.stdout[-1000:].strip())
    if result.returncode != 0 and result.stderr:
        print(f"WARN: {result.stderr[-500:].strip()}")
    return result

REPO_DIR = "/content/FLOTTE-FERRUS"

if not os.path.exists(REPO_DIR):
    # Premier clone
    print("[OSSEUS] Clone initial du repo...")
    if GITHUB_TOKEN:
        run(f"git clone https://{GITHUB_TOKEN}@github.com/{GITHUB_REPO}.git {REPO_DIR}")
    else:
        run(f"git clone https://github.com/{GITHUB_REPO}.git {REPO_DIR}")
else:
    # Git pull — recupere les mises a jour du code
    print("[OSSEUS] Mise a jour du codebase (git pull)...")
    if GITHUB_TOKEN:
        run(f"git remote set-url origin https://{GITHUB_TOKEN}@github.com/{GITHUB_REPO}.git",
            cwd=REPO_DIR, silent=True)
    run("git pull --rebase", cwd=REPO_DIR)

# Copier osseus_core.py depuis le repo vers Drive/codebase
# Le Drive garde une copie, le repo est la source de verite
src_script = f"{REPO_DIR}/04_FERRUS_OSSEUS/codebase/osseus_core.py"
run(f"cp '{src_script}' '{SCRIPT_PATH}'")
print(f"[OSSEUS] osseus_core.py synchronise -> {SCRIPT_PATH}")

# Verifier
if os.path.exists(SCRIPT_PATH):
    size = os.path.getsize(SCRIPT_PATH)
    print(f"[OSSEUS] Script pret ({size:,} octets)")
else:
    raise RuntimeError(f"Script introuvable : {SCRIPT_PATH}")

In [ ]:
# ══════════════════════════════════════════════════════════════════════════
# CELLULE 4 — INSTALLATION BLENDER
# ══════════════════════════════════════════════════════════════════════════

import subprocess

blender_check = subprocess.run("blender --version", shell=True, capture_output=True, text=True)

if blender_check.returncode != 0:
    print("[OSSEUS] Installation Blender (premiere fois, ~2 min)...")
    run("apt-get install -y blender 2>&1 | tail -3")
    blender_check2 = subprocess.run("blender --version", shell=True, capture_output=True, text=True)
    if blender_check2.returncode != 0:
        run("snap install blender --classic 2>&1 | tail -3")
else:
    ver = blender_check.stdout.strip().split("\n")[0]
    print(f"[OSSEUS] Blender : {ver}")

run("blender --version")

In [ ]:
# ══════════════════════════════════════════════════════════════════════════
# CELLULE 5 — VERIFICATION INPUT
# Upload si absent, confirmation si present
# ══════════════════════════════════════════════════════════════════════════

import os
from pathlib import Path

input_full_path = os.path.join(IN_DIR, INPUT_FILE)

if not os.path.exists(input_full_path):
    print(f"[OSSEUS] Fichier '{INPUT_FILE}' non trouve dans IN/")
    print(f"[OSSEUS] IN/ = {IN_DIR}")
    print("[OSSEUS] Upload via panneau Files Colab, ou coller directement dans Drive.")
    print()
    
    # Upload interactif
    from google.colab import files
    uploaded = files.upload()
    for fname, data in uploaded.items():
        dest = os.path.join(IN_DIR, fname)
        with open(dest, 'wb') as f:
            f.write(data)
        size_mb = len(data) / (1024*1024)
        print(f"[OSSEUS] Upload -> Drive/IN/{fname} ({size_mb:.2f} Mo)")
        # Auto-detect input file name
        if len(uploaded) == 1:
            INPUT_FILE      = fname
            input_full_path = dest
else:
    size_mb = os.path.getsize(input_full_path) / (1024*1024)
    ext     = Path(INPUT_FILE).suffix.lower()
    print(f"[OSSEUS] Input trouve : {INPUT_FILE} ({size_mb:.2f} Mo, {ext})")

In [ ]:
# ══════════════════════════════════════════════════════════════════════════
# CELLULE 6 — PIPELINE OSSEUS
# Rigging automatique via Blender headless
# ══════════════════════════════════════════════════════════════════════════

import os, subprocess, json
from pathlib import Path

stem        = Path(INPUT_FILE).stem
output_file = f"avatar_rigged_{stem}_{TEMPLATE}.fbx"
output_path = os.path.join(OUT_DIR, output_file)
report_path = os.path.join(OUT_DIR, f"rapport_osseus_{stem}.json")

print(f"[OSSEUS] ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━")
print(f"[OSSEUS] DEBUT PIPELINE")
print(f"[OSSEUS]   Input    : {INPUT_FILE}")
print(f"[OSSEUS]   Template : {TEMPLATE}")
print(f"[OSSEUS]   Sortie   : {output_file}")
print(f"[OSSEUS] ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━")

cmd = (
    f"blender --background --python '{SCRIPT_PATH}' -- "
    f"--input   '{input_full_path}' "
    f"--output  '{output_path}' "
    f"--template {TEMPLATE} "
    f"--report  '{report_path}'"
)

result = subprocess.run(cmd, shell=True, capture_output=True, text=True)

# Afficher uniquement les logs OSSEUS
all_output  = result.stdout + result.stderr
osseus_logs = [l for l in all_output.split("\n") if l.startswith("[OSSEUS]")]
print("\n".join(osseus_logs))

if result.returncode != 0 and not os.path.exists(output_path):
    print("\n[OSSEUS] ERREUR — Logs complets Blender :")
    print(all_output[-3000:])
    raise RuntimeError("Pipeline OSSEUS echoue")

In [ ]:
# ══════════════════════════════════════════════════════════════════════════
# CELLULE 7 — RAPPORT ET VALIDATION
# ══════════════════════════════════════════════════════════════════════════

import json, os

if os.path.exists(report_path):
    with open(report_path) as f:
        rapport = json.load(f)

    status = rapport.get('status', 'UNKNOWN')
    ok     = status == 'SUCCESS'

    print(f"[OSSEUS] ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━")
    print(f"[OSSEUS] RAPPORT — {'OK' if ok else 'ECHEC'} {status}")
    print(f"[OSSEUS] ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━")
    print(f"  Input         : {rapport.get('input', '-')}")
    print(f"  Template      : {rapport.get('template', '-')}")
    print(f"  Vertices      : {rapport.get('vertices', 0):,}")
    print(f"  Faces         : {rapport.get('faces', 0):,}")
    print(f"  Hauteur mesh  : {rapport.get('height', '-')}")
    print(f"  Largeur mesh  : {rapport.get('width', '-')}")
    print(f"  Bones crees   : {rapport.get('bones_count', '-')}")
    print(f"  Auto Weights  : {rapport.get('auto_weights', '-')}")
    print(f"  Taille sortie : {rapport.get('output_size_mb', '-')} Mo")

    if not ok:
        print(f"\n  ERREUR : {rapport.get('error', 'inconnue')}")
else:
    print("[OSSEUS] Rapport JSON introuvable")

# Confirmer que le FBX est sur Drive
if os.path.exists(output_path):
    size = os.path.getsize(output_path) / (1024*1024)
    print(f"\n[OSSEUS] FBX rige sur Drive : OUT/{output_file} ({size:.2f} Mo)")
    print(f"[OSSEUS] Prochaine etape : copier dans FERRUS ANIMUS IN/")
else:
    print("\n[OSSEUS] ERREUR : FBX non genere")

In [ ]:
# ══════════════════════════════════════════════════════════════════════════
# CELLULE 8 — TRANSFERT VERS FERRUS ANIMUS (optionnel)
# Copie automatique du FBX rige dans l'IN/ d'ANIMUS
# ══════════════════════════════════════════════════════════════════════════

import os, shutil

TRANSFERT_VERS_ANIMUS = False  # Passer a True pour copie automatique

ANIMUS_IN_DIR = f"{DRIVE_BASE}/01_FERRUS_ANIMUS/IN"

if TRANSFERT_VERS_ANIMUS and os.path.exists(output_path):
    os.makedirs(ANIMUS_IN_DIR, exist_ok=True)
    dest = os.path.join(ANIMUS_IN_DIR, output_file)
    shutil.copy2(output_path, dest)
    print(f"[OSSEUS] Transfere vers FERRUS ANIMUS IN/ :")
    print(f"  {dest}")
    print(f"[OSSEUS] Ouvrir 01_FERRUS_ANIMUS/main_ferrus.ipynb pour continuer.")
else:
    print("[OSSEUS] Transfert auto desactive (TRANSFERT_VERS_ANIMUS=False)")
    print(f"[OSSEUS] Copie manuelle : {output_path}")
    print(f"[OSSEUS] Vers           : {ANIMUS_IN_DIR}/{output_file}")